# LP-LLM: QLoRA Fine-tuning on Kaggle / Google Colab

**ICPR 2026 Low-Resolution License Plate Recognition Challenge**

This notebook fine-tunes **Qwen2-VL-2B** with **QLoRA (4-bit)** to fit within a free **T4 16GB** GPU.

| Setting | Value |
|---------|-------|
| Base Model | Qwen2-VL-2B-Instruct |
| Quantization | 4-bit NF4 (QLoRA) |
| LoRA rank | 16 |
| Batch size | 1 |
| Grad accumulation | 16 |
| Estimated VRAM | ~10-12 GB |
| Target GPU | T4 (16GB) / V100 / A100 |

## 0. Environment Check & GPU Info

In [ ]:
import torch
import os

# Check GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {gpu_mem:.1f} GB")
else:
    raise RuntimeError("No GPU found! Enable GPU in Runtime > Change runtime type.")

## 1. Install Dependencies

In [ ]:
%%capture
!pip install -U transformers>=4.45.0 accelerate>=0.34.0
!pip install peft>=0.13.0 bitsandbytes>=0.44.0
!pip install qwen-vl-utils einops
!pip install editdistance tqdm wandb
!pip install Pillow numpy pandas

## 2. Upload & Prepare Dataset

**Option A — Kaggle:** Add the dataset as a Kaggle Dataset, it will appear at `/kaggle/input/`

**Option B — Colab:** Upload a zip or mount Google Drive

**Option C — HuggingFace Hub:** Upload dataset to a private HF dataset repo

In [ ]:
import platform

# ---------- Auto-detect platform ----------
if os.path.exists("/kaggle"):
    PLATFORM = "kaggle"
    # Kaggle: update the dataset slug below to match your upload
    DATA_ROOT = "/kaggle/input/icpr2026-lrlpr/data"
    OUTPUT_DIR = "/kaggle/working/outputs"
elif "google.colab" in str(globals().get("get_ipython", lambda: None)()):
    PLATFORM = "colab"
    DATA_ROOT = "/content/data"
    OUTPUT_DIR = "/content/outputs"
else:
    PLATFORM = "local"
    DATA_ROOT = "../data"
    OUTPUT_DIR = "./outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Platform: {PLATFORM}")
print(f"Data root: {DATA_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
# -------- Option B: Colab — mount Google Drive --------
# Uncomment if using Google Colab with data on Drive

# from google.colab import drive
# drive.mount('/content/drive')
# DATA_ROOT = '/content/drive/MyDrive/ICPR/data'

# -------- Option B: Colab — upload zip --------
# Uncomment if uploading a zip file

# !unzip -q /content/data.zip -d /content/data

In [ ]:
# Verify dataset structure
from pathlib import Path

data_path = Path(DATA_ROOT)
print(f"Data root exists: {data_path.exists()}")

if data_path.exists():
    for scenario in ["train/Scenario-A", "train/Scenario-B"]:
        sp = data_path / scenario
        if sp.exists():
            layouts = [d.name for d in sp.iterdir() if d.is_dir()]
            for layout in layouts:
                n_tracks = sum(1 for d in (sp / layout).iterdir() if d.is_dir())
                print(f"  {scenario}/{layout}: {n_tracks} tracks")

    test_dir = data_path / "Pa7a3Hin-test-public"
    if test_dir.exists():
        n_test = sum(1 for d in test_dir.iterdir() if d.is_dir())
        print(f"  Test: {n_test} tracks")

## 3. Configuration

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    # ----- Model -----
    model_name: str = "Qwen/Qwen2-VL-2B-Instruct"

    # ----- QLoRA -----
    use_qlora: bool = True           # 4-bit quantization
    bnb_4bit_quant_type: str = "nf4" # NormalFloat4
    bnb_4bit_compute_dtype: str = "bfloat16"
    lora_r: int = 16                 # Lower rank for T4
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ])

    # ----- Training -----
    num_epochs: int = 5
    batch_size: int = 1              # Must be 1 for T4
    gradient_accumulation_steps: int = 16
    effective_batch_size: int = 16   # batch_size * grad_accum
    learning_rate: float = 2e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    max_grad_norm: float = 1.0
    gradient_checkpointing: bool = True

    # ----- Data -----
    image_type: str = "lr"           # Train on low-res only
    val_ratio: float = 0.1
    max_samples: int = 0             # 0 = use all (set small number for debugging)

    # ----- Prompts -----
    system_prompt: str = (
        "You are an expert license plate recognition system. "
        "Analyze the given license plate image and output ONLY the plate text."
    )
    user_prompt: str = (
        "Recognize the license plate text in this image. "
        "Output only the characters on the plate, nothing else."
    )

    # ----- Logging -----
    log_every: int = 50
    eval_every_epoch: bool = True
    use_wandb: bool = False          # Set True if you have a wandb account
    wandb_project: str = "lp-llm-qlora"
    seed: int = 42

cfg = Config()

# Quick debug run — uncomment to test with 50 samples
# cfg.max_samples = 50
# cfg.num_epochs = 1

print("Config ready.")
print(f"  Model: {cfg.model_name}")
print(f"  QLoRA: {cfg.use_qlora} | LoRA r={cfg.lora_r} alpha={cfg.lora_alpha}")
print(f"  Batch: {cfg.batch_size} × {cfg.gradient_accumulation_steps} = {cfg.effective_batch_size}")
print(f"  Epochs: {cfg.num_epochs}")

## 4. Load Model with 4-bit Quantization

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=torch.bfloat16 if cfg.bnb_4bit_compute_dtype == "bfloat16" else torch.float16,
    bnb_4bit_use_double_quant=True,  # Nested quantization for extra savings
)

print(f"Loading {cfg.model_name} in 4-bit...")

# Load processor
processor = AutoProcessor.from_pretrained(
    cfg.model_name,
    trust_remote_code=True,
)

# Load model in 4-bit
model = Qwen2VLForConditionalGeneration.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=cfg.gradient_checkpointing,
)

print("Model loaded!")

# Print memory usage
mem_alloc = torch.cuda.memory_allocated() / 1e9
mem_reserved = torch.cuda.memory_reserved() / 1e9
print(f"GPU memory — allocated: {mem_alloc:.2f} GB | reserved: {mem_reserved:.2f} GB")

In [ ]:
# Apply LoRA adapters
lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=cfg.lora_target_modules,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Memory after LoRA
mem_alloc = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory after LoRA: {mem_alloc:.2f} GB")

## 5. Prepare Dataset

In [ ]:
import json
import random
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader

random.seed(cfg.seed)


class LPRDataset(Dataset):
    """Lightweight dataset for QLoRA training on Kaggle/Colab."""

    def __init__(self, samples, processor, system_prompt, user_prompt):
        self.samples = samples
        self.processor = processor
        self.system_prompt = system_prompt
        self.user_prompt = user_prompt

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample["image_path"]).convert("RGB")
        plate_text = sample["plate_text"]

        # Build conversation
        messages = [
            {"role": "system", "content": self.system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": self.user_prompt},
                ],
            },
        ]

        # Build prompt text (without answer)
        prompt_text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )

        # Full text with answer
        full_text = prompt_text + plate_text + self.processor.tokenizer.eos_token

        # Tokenize prompt (for masking)
        prompt_inputs = self.processor(
            text=[prompt_text], images=[image],
            return_tensors="pt", padding=False,
        )
        prompt_len = prompt_inputs["input_ids"].shape[1]

        # Tokenize full sequence
        full_inputs = self.processor(
            text=[full_text], images=[image],
            return_tensors="pt", padding=False,
        )

        input_ids = full_inputs["input_ids"].squeeze(0)
        attention_mask = full_inputs["attention_mask"].squeeze(0)

        # Create labels: mask prompt tokens with -100
        labels = input_ids.clone()
        labels[:prompt_len] = -100

        result = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

        # Include pixel values and image grid if present
        if "pixel_values" in full_inputs:
            result["pixel_values"] = full_inputs["pixel_values"].squeeze(0)
        if "image_grid_thw" in full_inputs:
            result["image_grid_thw"] = full_inputs["image_grid_thw"].squeeze(0)

        return result


def load_samples(data_root, image_type="lr", max_samples=0):
    """Load all training samples from the ICPR data structure."""
    samples = []
    data_path = Path(data_root) / "train"

    for scenario in ["Scenario-A", "Scenario-B"]:
        for layout in ["Brazilian", "Mercosur"]:
            layout_dir = data_path / scenario / layout
            if not layout_dir.exists():
                continue

            for track_dir in layout_dir.iterdir():
                if not track_dir.is_dir() or track_dir.name.startswith("."):
                    continue

                ann_path = track_dir / "annotations.json"
                if not ann_path.exists():
                    continue

                with open(ann_path) as f:
                    ann = json.load(f)

                prefix = "lr" if image_type == "lr" else "hr"
                for img_path in sorted(track_dir.glob(f"{prefix}-*.png")) + \
                                sorted(track_dir.glob(f"{prefix}-*.jpg")):
                    samples.append({
                        "image_path": str(img_path),
                        "plate_text": ann["plate_text"],
                        "plate_layout": ann.get("plate_layout", layout),
                        "track_id": track_dir.name,
                    })

    random.shuffle(samples)
    if max_samples > 0:
        samples = samples[:max_samples]

    return samples


def split_by_track(samples, val_ratio=0.1):
    """Split samples by track to avoid data leakage."""
    track_ids = list(set(s["track_id"] for s in samples))
    random.shuffle(track_ids)

    split_idx = int(len(track_ids) * (1 - val_ratio))
    train_tracks = set(track_ids[:split_idx])
    val_tracks = set(track_ids[split_idx:])

    train_samples = [s for s in samples if s["track_id"] in train_tracks]
    val_samples = [s for s in samples if s["track_id"] in val_tracks]

    return train_samples, val_samples


# Load and split
print("Loading samples...")
all_samples = load_samples(DATA_ROOT, image_type=cfg.image_type, max_samples=cfg.max_samples)
train_samples, val_samples = split_by_track(all_samples, cfg.val_ratio)

print(f"Total: {len(all_samples)} | Train: {len(train_samples)} | Val: {len(val_samples)}")
print(f"Sample plate text: {train_samples[0]['plate_text']}")

In [ ]:
def collate_fn(batch):
    """Pad sequences to the same length within a batch."""
    pad_token_id = processor.tokenizer.pad_token_id or 0

    max_len = max(item["input_ids"].shape[0] for item in batch)

    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for item in batch:
        seq_len = item["input_ids"].shape[0]
        pad_len = max_len - seq_len

        input_ids_list.append(
            torch.cat([item["input_ids"], torch.full((pad_len,), pad_token_id)])
        )
        attention_mask_list.append(
            torch.cat([item["attention_mask"], torch.zeros(pad_len, dtype=torch.long)])
        )
        labels_list.append(
            torch.cat([item["labels"], torch.full((pad_len,), -100)])
        )

    result = {
        "input_ids": torch.stack(input_ids_list),
        "attention_mask": torch.stack(attention_mask_list),
        "labels": torch.stack(labels_list),
    }

    # Pixel values — concatenate along token dim (Qwen2-VL format)
    if "pixel_values" in batch[0]:
        result["pixel_values"] = torch.cat([item["pixel_values"] for item in batch], dim=0)
    if "image_grid_thw" in batch[0]:
        result["image_grid_thw"] = torch.cat([item["image_grid_thw"].unsqueeze(0)
                                               if item["image_grid_thw"].dim() == 1
                                               else item["image_grid_thw"]
                                               for item in batch], dim=0)

    return result


# Create datasets and loaders
train_dataset = LPRDataset(train_samples, processor, cfg.system_prompt, cfg.user_prompt)
val_dataset = LPRDataset(val_samples, processor, cfg.system_prompt, cfg.user_prompt)

train_loader = DataLoader(
    train_dataset, batch_size=cfg.batch_size, shuffle=True,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=cfg.batch_size, shuffle=False,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 6. Training Loop

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from tqdm.auto import tqdm
import time

if cfg.use_wandb:
    import wandb
    wandb.init(project=cfg.wandb_project, config=vars(cfg))

# Optimizer — only train LoRA params
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=cfg.learning_rate, weight_decay=cfg.weight_decay)

# Scheduler
num_training_steps = (len(train_loader) // cfg.gradient_accumulation_steps) * cfg.num_epochs
num_warmup_steps = int(num_training_steps * cfg.warmup_ratio)

warmup_sched = LinearLR(optimizer, start_factor=0.1, total_iters=num_warmup_steps)
cosine_sched = CosineAnnealingLR(optimizer, T_max=num_training_steps - num_warmup_steps, eta_min=cfg.learning_rate * 0.1)
scheduler = SequentialLR(optimizer, [warmup_sched, cosine_sched], milestones=[num_warmup_steps])

print(f"Training steps: {num_training_steps} | Warmup: {num_warmup_steps}")

In [ ]:
device = torch.device("cuda")
best_val_loss = float("inf")
global_step = 0
train_start = time.time()

for epoch in range(cfg.num_epochs):
    # ==================== TRAIN ====================
    model.train()
    epoch_loss = 0
    num_batches = 0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.num_epochs}")

    for step, batch in enumerate(pbar):
        # Move batch to device
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                 for k, v in batch.items()}

        # Forward
        outputs = model(**batch)
        loss = outputs.loss / cfg.gradient_accumulation_steps

        # Backward
        loss.backward()

        # Optimizer step every N accumulation steps
        if (step + 1) % cfg.gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

        epoch_loss += loss.item() * cfg.gradient_accumulation_steps
        num_batches += 1

        # Progress bar
        pbar.set_postfix({
            "loss": f"{epoch_loss / num_batches:.4f}",
            "lr": f"{scheduler.get_last_lr()[0]:.2e}",
            "mem": f"{torch.cuda.memory_allocated()/1e9:.1f}G",
        })

        # Log
        if cfg.use_wandb and global_step % cfg.log_every == 0 and global_step > 0:
            wandb.log({"train/loss": epoch_loss / num_batches,
                       "train/lr": scheduler.get_last_lr()[0],
                       "train/step": global_step})

    avg_train_loss = epoch_loss / num_batches

    # ==================== VALIDATE ====================
    model.eval()
    val_loss = 0
    val_batches = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating", leave=False):
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                     for k, v in batch.items()}

            outputs = model(**batch)
            val_loss += outputs.loss.item()
            val_batches += 1

    avg_val_loss = val_loss / val_batches if val_batches > 0 else float("inf")

    elapsed = time.time() - train_start
    print(f"\nEpoch {epoch+1}/{cfg.num_epochs} — "
          f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
          f"Time: {elapsed/60:.1f}min")

    if cfg.use_wandb:
        wandb.log({"val/loss": avg_val_loss, "epoch": epoch + 1})

    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        save_path = os.path.join(OUTPUT_DIR, "best_model")
        model.save_pretrained(save_path)
        processor.save_pretrained(save_path)
        print(f"  Best model saved to {save_path}")

# Save final model
final_path = os.path.join(OUTPUT_DIR, "final_model")
model.save_pretrained(final_path)
processor.save_pretrained(final_path)
print(f"\nTraining complete! Final model saved to {final_path}")
print(f"Best val loss: {best_val_loss:.4f}")

if cfg.use_wandb:
    wandb.finish()

## 7. Evaluation

In [ ]:
import editdistance


@torch.no_grad()
def generate_prediction(model, processor, image, system_prompt, user_prompt):
    """Generate plate text prediction for a single image."""
    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": user_prompt},
            ],
        },
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )

    inputs = processor(
        text=[text], images=[image], return_tensors="pt", padding=True,
    )
    inputs = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v
              for k, v in inputs.items()}

    generated_ids = model.generate(
        **inputs, max_new_tokens=15, num_beams=1, do_sample=False,
    )

    # Decode only the generated part
    input_len = inputs["input_ids"].shape[1]
    output = processor.batch_decode(
        generated_ids[:, input_len:], skip_special_tokens=True,
    )[0].strip()

    # Keep only alphanumeric
    output = "".join(c for c in output.upper() if c.isalnum())

    return output


def evaluate_samples(model, processor, samples, system_prompt, user_prompt, max_eval=200):
    """Evaluate model on a subset of samples."""
    model.eval()

    eval_samples = samples[:max_eval]
    predictions = []
    targets = []

    for sample in tqdm(eval_samples, desc="Evaluating"):
        image = Image.open(sample["image_path"]).convert("RGB")
        pred = generate_prediction(model, processor, image, system_prompt, user_prompt)

        predictions.append(pred)
        targets.append(sample["plate_text"])

    # Compute metrics
    correct = sum(1 for p, t in zip(predictions, targets) if p == t)
    accuracy = correct / len(predictions)

    total_ed = sum(editdistance.eval(p, t) for p, t in zip(predictions, targets))
    avg_ed = total_ed / len(predictions)

    total_char_correct = 0
    total_chars = 0
    for p, t in zip(predictions, targets):
        max_len = max(len(p), len(t))
        for i in range(min(len(p), len(t))):
            if p[i] == t[i]:
                total_char_correct += 1
        total_chars += max_len

    char_acc = total_char_correct / total_chars if total_chars > 0 else 0

    print(f"\n{'='*50}")
    print(f"EVALUATION RESULTS ({len(eval_samples)} samples)")
    print(f"{'='*50}")
    print(f"  Accuracy:           {accuracy:.4f} ({correct}/{len(predictions)})")
    print(f"  Character Accuracy: {char_acc:.4f}")
    print(f"  Avg Edit Distance:  {avg_ed:.4f}")
    print(f"{'='*50}")

    # Show some examples
    print(f"\nSample predictions:")
    for i in range(min(10, len(predictions))):
        match = "✓" if predictions[i] == targets[i] else "✗"
        print(f"  {match} GT: {targets[i]:>10} | Pred: {predictions[i]:>10}")

    return {"accuracy": accuracy, "char_accuracy": char_acc, "edit_distance": avg_ed}


# Run evaluation
metrics = evaluate_samples(
    model, processor, val_samples,
    cfg.system_prompt, cfg.user_prompt,
    max_eval=200,
)

## 8. Multi-Image Fusion (MVCP)

In [ ]:
from collections import Counter, defaultdict


def mvcp_fusion(predictions):
    """Majority Vote by Character Position."""
    if not predictions:
        return ""

    max_len = max(len(p) for p in predictions)
    result = []

    for pos in range(max_len):
        chars = [p[pos] for p in predictions if pos < len(p)]
        if chars:
            result.append(Counter(chars).most_common(1)[0][0])

    return "".join(result)


def evaluate_tracks_with_fusion(model, processor, data_root, system_prompt, user_prompt, max_tracks=50):
    """Evaluate using multi-image fusion on tracks."""
    model.eval()

    data_path = Path(data_root) / "train"
    tracks_evaluated = 0
    correct_single = 0
    correct_mvcp = 0

    for scenario in ["Scenario-A", "Scenario-B"]:
        for layout in ["Brazilian", "Mercosur"]:
            layout_dir = data_path / scenario / layout
            if not layout_dir.exists():
                continue

            for track_dir in sorted(layout_dir.iterdir()):
                if tracks_evaluated >= max_tracks:
                    break
                if not track_dir.is_dir() or track_dir.name.startswith("."):
                    continue

                ann_path = track_dir / "annotations.json"
                if not ann_path.exists():
                    continue

                with open(ann_path) as f:
                    ann = json.load(f)

                gt = ann["plate_text"]

                # Get all LR images
                lr_images = sorted(track_dir.glob("lr-*.png")) + sorted(track_dir.glob("lr-*.jpg"))
                if not lr_images:
                    continue

                # Predict each image
                preds = []
                for img_path in lr_images:
                    image = Image.open(img_path).convert("RGB")
                    pred = generate_prediction(model, processor, image, system_prompt, user_prompt)
                    preds.append(pred)

                # Single-image (first)
                if preds[0] == gt:
                    correct_single += 1

                # MVCP fusion
                fused = mvcp_fusion(preds)
                if fused == gt:
                    correct_mvcp += 1

                tracks_evaluated += 1

            if tracks_evaluated >= max_tracks:
                break
        if tracks_evaluated >= max_tracks:
            break

    print(f"\n{'='*50}")
    print(f"TRACK-LEVEL EVALUATION ({tracks_evaluated} tracks)")
    print(f"{'='*50}")
    print(f"  Single-image accuracy: {correct_single/tracks_evaluated:.4f}")
    print(f"  MVCP fusion accuracy:  {correct_mvcp/tracks_evaluated:.4f}")
    print(f"  Improvement:           {(correct_mvcp - correct_single)/tracks_evaluated:+.4f}")
    print(f"{'='*50}")


# Uncomment to run track-level evaluation (takes longer)
# evaluate_tracks_with_fusion(model, processor, DATA_ROOT, cfg.system_prompt, cfg.user_prompt, max_tracks=50)

## 9. Generate Test Submission

In [ ]:
import csv


def generate_submission(model, processor, data_root, output_dir, system_prompt, user_prompt):
    """Generate submission CSV for ICPR 2026 challenge."""
    model.eval()

    test_dir = Path(data_root) / "Pa7a3Hin-test-public"
    if not test_dir.exists():
        print(f"Test directory not found: {test_dir}")
        return

    results = {}

    track_dirs = sorted([d for d in test_dir.iterdir() if d.is_dir() and not d.name.startswith(".")])
    print(f"Processing {len(track_dirs)} test tracks...")

    for track_dir in tqdm(track_dirs, desc="Test inference"):
        lr_images = sorted(track_dir.glob("lr-*.jpg")) + sorted(track_dir.glob("lr-*.png"))

        if not lr_images:
            continue

        # Predict each image
        preds = []
        for img_path in lr_images:
            image = Image.open(img_path).convert("RGB")
            pred = generate_prediction(model, processor, image, system_prompt, user_prompt)
            preds.append(pred)

        # MVCP fusion
        fused = mvcp_fusion(preds)
        results[track_dir.name] = fused

    # Write submission CSV
    os.makedirs(output_dir, exist_ok=True)
    submission_path = os.path.join(output_dir, "submission.csv")

    with open(submission_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "plate_text"])
        for track_id, prediction in sorted(results.items()):
            clean_id = track_id.replace("track_", "")
            writer.writerow([clean_id, prediction])

    print(f"\nSubmission saved to {submission_path}")
    print(f"Total predictions: {len(results)}")

    # Show first 10 predictions
    print("\nSample predictions:")
    for track_id, pred in list(sorted(results.items()))[:10]:
        print(f"  {track_id}: {pred}")


# Uncomment to generate submission
# generate_submission(model, processor, DATA_ROOT, OUTPUT_DIR, cfg.system_prompt, cfg.user_prompt)

## 10. Save & Download

### Kaggle
Models saved to `/kaggle/working/outputs/` are available in the Output tab after the notebook finishes.

### Colab
Use the cells below to download or push to Hugging Face Hub.

In [ ]:
# Option A: Download from Colab
# from google.colab import files
# !zip -r /content/lp_llm_qlora.zip {OUTPUT_DIR}/best_model
# files.download("/content/lp_llm_qlora.zip")

# Option B: Push LoRA adapters to Hugging Face Hub
# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")
# model.push_to_hub("your-username/lp-llm-qlora-2b")
# processor.push_to_hub("your-username/lp-llm-qlora-2b")

# Option C: Copy to Google Drive (Colab)
# !cp -r {OUTPUT_DIR}/best_model /content/drive/MyDrive/ICPR/best_model

print("Done! Saved artifacts:")
print(f"  Best model: {OUTPUT_DIR}/best_model")
print(f"  Final model: {OUTPUT_DIR}/final_model")

In [ ]:
# Final memory report
if torch.cuda.is_available():
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
    print(f"Current GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")